# ChaoSSM — Phase 1: Synthetic Task Validation

## Why this phase, now

Phase 0 (a–d) answered whether the chaotic mechanism is *controllable*: can a learnable map
reach the edge-of-chaos corridor, does the estimator track ground truth, is the constraint
holdable. By 0d, yes to all three, with a broad performance/stability plateau at λ ∈ [0, 0.4]
and both tails (λ=−0.2, λ=0.65) significantly worse.

**What has never been tested: whether chaos helps at all**, relative to a linear recurrence.
Every 0a–0d comparison was chaotic-variant-vs-chaotic-variant. This phase adds the comparison
the whole proposal's novelty claim depends on — §2.1's argument that linear SSMs (S4 → Mamba-3)
sit on a structural ceiling that a chaotic core can exceed.

## Design

**Three arms, identical everywhere except the recurrence nonlinearity** — same gating, same
`A`/`B`/`W_g`, same state dimension, same optimizer. Only `φ` differs, so any gap is attributable
to chaos rather than confounded by architecture:

- **`linear`** — `φ(h) = W_lin · h`, no fold, no `r`. The ceiling the proposal argues against.
- **`chaossm_free`** — the 0d tunable-gain fold, unconstrained. Tied with the best explicit
  target in 0d.
- **`chaossm_corridor`** — 0d's dual ascent, held near λ=0.15 (0d's empirical optimum).

**Four tasks**, from proposal §5.2, at a harder setting than Phase 0's single copy task:

- **Copying** — reproduce L=12 random tokens after a delimiter.
- **Selective Copying** — L=24 positions, only M=6 are content tokens (rest are a `blank` filler);
  reproduce the M content tokens in order, ignoring blanks.
- **Associative Recall** — M=6 (key, value) pairs, then a query key; output the paired value.
  Single-token target, not full-sequence.
- **Long-Range Parity** — L=48 tokens, exactly 2 positions carry relevant bits scattered far
  apart, the rest are noise; output the XOR of the two relevant bits. Unsolvable by local
  pattern matching — the task Phase 0's copy task couldn't probe.

Instrumentation and statistical protocol (paired Wilcoxon, 5 seeds, min achievable p reported)
follow the standard established in 0a–0d.

## Setup

In [1]:
import os, json, math, time, warnings
from dataclasses import dataclass, asdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import wilcoxon
from torch.func import jacrev

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE)

def setup_results_dir(project="ChaoSSM", phase="phase1"):
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        base = f"/content/drive/My Drive/{project}/results/{phase}"
    except Exception as e:
        base = os.path.abspath(f"./{project}_results/{phase}")
        print(f"[info] Not in Colab ({type(e).__name__}); using local dir.")
    os.makedirs(base, exist_ok=True); print("results ->", base); return base

RESULTS = setup_results_dir()
def savefig(f,n): p=os.path.join(RESULTS,n); f.savefig(p,dpi=300,bbox_inches="tight"); print("saved",p)
def savetable(d,n): p=os.path.join(RESULTS,n); d.to_csv(p,index=False); print("saved",p)
def savejson(o,n):
    p=os.path.join(RESULTS,n)
    with open(p,"w") as fh: json.dump(o,fh,indent=2,default=str)
    print("saved",p)

torch 2.11.0+cu128 | device: cuda
Mounted at /content/drive
results -> /content/drive/My Drive/ChaoSSM/results/phase1


## Configuration

Steps are reduced from 0d's 2000 to 1200: this phase runs 3 arms × 4 tasks × 5 seeds = 60 runs
(vs. 0d's 30), so per-run budget is trimmed to keep total wall-clock comparable. `lyap_subsample`
is also cut for the same reason — only the two chaotic arms need it at all.

In [2]:
SMOKE_TEST = False   # full run: 3 arms x 4 tasks x 5 seeds

@dataclass
class Config:
    state_dim: int = 48
    d_model: int = 48
    vocab: int = 16
    steps: int = 1200
    batch: int = 64
    lr: float = 3e-4
    bptt_k: int = 64
    grad_clip: float = 1.0
    # chaotic map (carried over from 0d, settled)
    gamma_init: float = 1.0
    warmup_frac: float = 0.10
    dual_lr: float = 0.05
    alpha_max: float = 50.0
    corridor_target: float = 0.15     # 0d's empirical optimum
    corridor_half_width: float = 0.10
    lyap_window: int = 4
    lyap_subsample: int = 3
    lyap_every: int = 20
    benettin_transient: int = 100
    benettin_measure: int = 250
    # tasks
    copy_len: int = 12
    sel_len: int = 24
    sel_content: int = 6
    recall_pairs: int = 6
    parity_len: int = 48
    # experiment
    arms: tuple = ("linear", "chaossm_free", "chaossm_corridor")
    tasks: tuple = ("copy", "selective_copy", "assoc_recall", "parity")
    seeds: int = 5
    eval_every: int = 20

CFG = Config()
if SMOKE_TEST:
    CFG.steps = 150; CFG.seeds = 1
    CFG.tasks = ("copy", "parity")
    CFG.benettin_measure = 60; CFG.benettin_transient = 20
    print("*** SMOKE TEST — pipeline check only ***")
print(json.dumps({k: str(v) for k, v in asdict(CFG).items()}, indent=2))

{
  "state_dim": "48",
  "d_model": "48",
  "vocab": "16",
  "steps": "1200",
  "batch": "64",
  "lr": "0.0003",
  "bptt_k": "64",
  "grad_clip": "1.0",
  "gamma_init": "1.0",
  "warmup_frac": "0.1",
  "dual_lr": "0.05",
  "alpha_max": "50.0",
  "corridor_target": "0.15",
  "corridor_half_width": "0.1",
  "lyap_window": "4",
  "lyap_subsample": "3",
  "lyap_every": "20",
  "benettin_transient": "100",
  "benettin_measure": "250",
  "copy_len": "12",
  "sel_len": "24",
  "sel_content": "6",
  "recall_pairs": "6",
  "parity_len": "48",
  "arms": "('linear', 'chaossm_free', 'chaossm_corridor')",
  "tasks": "('copy', 'selective_copy', 'assoc_recall', 'parity')",
  "seeds": "5",
  "eval_every": "20"
}


## 1. Model — shared cell, pluggable recurrence

`LinearMap` and `ChaoticMap` are interchangeable `phi` modules dropped into the identical
`ChaoSSMCell`. This is the fairness guarantee: parameter count, gating, and everything else
is held fixed across arms.

In [3]:
def triangle_fold(x):
    return 1.0 - torch.abs(torch.remainder(x, 2.0) - 1.0)


class LinearMap(nn.Module):
    '''The baseline: phi(h) = W_lin @ h. No chaos, no fold -- the linear-SSM ceiling.'''
    def __init__(self, N):
        super().__init__()
        W = torch.empty(N, N); nn.init.orthogonal_(W)
        self.W_lin = nn.Parameter(W * 0.5)   # scaled so its operating norm is comparable
    def forward(self, h):
        return torch.tanh(h @ self.W_lin.T)  # bounded, matching phi's (0,1)-ish range via tanh


class ChaoticMap(nn.Module):
    '''phi(h) = fold( gamma * (W_mix h + r h(1-h)) ) -- settled design from 0d.
    sigma(J_phi) = gamma * sigma(affine part) exactly (fold is orthogonal on derivatives).'''
    def __init__(self, N, gamma_init=1.0):
        super().__init__()
        self.w_r = nn.Parameter(torch.zeros(N))
        W = torch.empty(N, N); nn.init.orthogonal_(W)
        self.W_mix = nn.Parameter(W * 0.1)
        inv = math.log(math.expm1(gamma_init))
        self.w_gamma = nn.Parameter(torch.tensor(float(inv)))
    def get_r(self): return 4.0 * torch.sigmoid(self.w_r)
    def get_gamma(self): return F.softplus(self.w_gamma)
    def forward(self, h):
        r = self.get_r()
        u = h @ self.W_mix.T + r * h * (1 - h)
        return triangle_fold(self.get_gamma() * u)


class ChaoSSMCell(nn.Module):
    def __init__(self, cfg, phi_kind):
        super().__init__()
        self.N = cfg.state_dim
        self.phi_kind = phi_kind
        self.phi = LinearMap(cfg.state_dim) if phi_kind == "linear" \
                   else ChaoticMap(cfg.state_dim, cfg.gamma_init)
        self.A = nn.Parameter(torch.diag(torch.empty(cfg.state_dim).uniform_(0.9, 1.1)))
        self.B = nn.Parameter(torch.randn(cfg.state_dim, cfg.d_model) / math.sqrt(cfg.d_model))
        self.W_g = nn.Parameter(torch.randn(cfg.state_dim, cfg.d_model) / math.sqrt(cfg.d_model))
        self.b_g = nn.Parameter(torch.zeros(cfg.state_dim))
    def step(self, h, x):
        g = torch.sigmoid(x @ self.W_g.T + self.b_g)
        raw = g * self.phi(h @ self.A.T) + (1 - g) * h + x @ self.B.T
        return triangle_fold(raw) if self.phi_kind != "linear" else torch.tanh(raw)
    def init_state(self, b, dev):
        return torch.zeros(b, self.N, device=dev) if self.phi_kind == "linear" \
               else torch.full((b, self.N), 0.5, device=dev)


class ChaoSSMLM(nn.Module):
    '''vocab_size covers whatever the task needs; readout_size may differ from vocab_size
    (e.g. parity has 2 output classes).'''
    def __init__(self, cfg, phi_kind, vocab_size, readout_size):
        super().__init__()
        self.cfg = cfg
        self.emb = nn.Embedding(vocab_size, cfg.d_model)
        self.cell = ChaoSSMCell(cfg, phi_kind)
        self.readout = nn.Linear(cfg.state_dim, readout_size)
    def forward(self, tok, return_traj=False):
        B, T = tok.shape
        x = self.emb(tok); h = self.cell.init_state(B, tok.device)
        outs, traj = [], []
        for t in range(T):
            h = self.cell.step(h, x[:, t]); outs.append(self.readout(h))
            if return_traj: traj.append(h)
            if (t + 1) % self.cfg.bptt_k == 0: h = h.detach()
        lg = torch.stack(outs, 1)
        return (lg, torch.stack(traj, 1), x) if return_traj else lg

## 2. Estimators (carried over from 0d, unchanged; validated there to slope 1.0000)

In [4]:
def _full_jac_fn(model, x):
    return lambda z: model.cell.step(z.unsqueeze(0), x.unsqueeze(0))[0]

def lambda_window(model, traj, xseq, W, n_sample):
    B, T, N = traj.shape
    if T <= W + 1:
        return torch.zeros((), device=traj.device)
    lams = []
    for b in torch.randperm(B)[:n_sample]:
        t0 = int(torch.randint(0, T - W - 1, (1,)))
        M = torch.eye(N, device=traj.device, dtype=traj.dtype)
        for k in range(W):
            M = jacrev(_full_jac_fn(model, xseq[b, t0+k]))(traj[b, t0+k]) @ M
        lams.append(torch.log(torch.linalg.matrix_norm(M, ord=2) + 1e-12) / W)
    return torch.stack(lams).mean()

@torch.no_grad()
def benettin_spectrum(step_fn, h0, inputs, N, T_tr, T_me):
    h = h0.clone()
    for t in range(T_tr): h = step_fn(h, inputs[t % len(inputs)])
    Q = torch.eye(N, device=h.device, dtype=h.dtype); lyap = torch.zeros(N, device=h.device, dtype=h.dtype)
    for t in range(T_me):
        x = inputs[(T_tr + t) % len(inputs)]
        Q = jacrev(lambda hh: step_fn(hh, x))(h) @ Q
        Q, R = torch.linalg.qr(Q)
        lyap += torch.log(torch.abs(torch.diagonal(R)) + 1e-12)
        h = step_fn(h, x)
    return (lyap / T_me).sort(descending=True).values

def kaplan_yorke(sp):
    s = sp.detach().cpu().numpy().astype(np.float64); c = np.cumsum(s)
    idx = np.where(c >= 0)[0]
    if len(idx) == 0: return 0.0
    j = int(idx[-1]) + 1
    if j >= len(s): return float(len(s))
    d = abs(s[j])
    return float(j) if d < 1e-12 else float(j + c[j-1]/d)

## 3. Task generators

Each returns `(input_ids, target_ids, loss_mask, vocab_size, readout_size, metric_fn)`. Full-
sequence tasks (copy, selective copy) mask everything before the answer region; single-token
tasks (recall, parity) mask everything except the final position. `metric_fn(logits, targets,
mask)` returns an exact-match accuracy so all four tasks report on the same scale.

In [5]:
def make_copy_batch(cfg, batch, device, gen):
    L, V = cfg.copy_len, cfg.vocab
    sep, pad = V, V + 1
    src = torch.randint(0, V, (batch, L), device=device, generator=gen)
    inp = torch.cat([src, torch.full((batch,1), sep, device=device),
                     torch.full((batch,L), pad, device=device)], 1)
    tgt = torch.cat([torch.full((batch,L+1), -100, device=device), src], 1)
    return inp, tgt

def make_selective_copy_batch(cfg, batch, device, gen):
    '''L positions, only M carry content; rest are `blank`. Reproduce the M content
    tokens in order after a delimiter.'''
    L, M, V = cfg.sel_len, cfg.sel_content, cfg.vocab
    blank, sep, pad = V, V + 1, V + 2
    seq = torch.full((batch, L), blank, device=device)
    content = torch.randint(0, V, (batch, M), device=device, generator=gen)
    for b in range(batch):
        pos = torch.randperm(L, generator=gen, device=device)[:M].sort().values
        seq[b, pos] = content[b]
    inp = torch.cat([seq, torch.full((batch,1), sep, device=device),
                     torch.full((batch,M), pad, device=device)], 1)
    tgt = torch.cat([torch.full((batch, L+1), -100, device=device), content], 1)
    return inp, tgt

def make_assoc_recall_batch(cfg, batch, device, gen):
    '''M (key,value) pairs, then a query key -> output the paired value at the final position.'''
    M, V = cfg.recall_pairs, cfg.vocab
    sep = V
    keys = torch.stack([torch.randperm(V, generator=gen, device=device)[:M] for _ in range(batch)])
    vals = torch.randint(0, V, (batch, M), device=device, generator=gen)
    qidx = torch.randint(0, M, (batch,), device=device, generator=gen)
    bidx = torch.arange(batch, device=device)
    qkey = keys[bidx, qidx]
    qval = vals[bidx, qidx]
    pairs = torch.stack([keys, vals], dim=2).reshape(batch, 2*M)   # k1,v1,k2,v2,...
    inp = torch.cat([pairs, torch.full((batch,1), sep, device=device), qkey.unsqueeze(1)], 1)
    tgt = torch.full((batch, inp.shape[1]), -100, device=device)
    tgt[:, -1] = qval
    return inp, tgt

def make_parity_batch(cfg, batch, device, gen):
    '''L positions, 2 carry relevant bits (encoded as tokens 0/1), rest are noise
    tokens (>=2). Target = XOR of the two relevant bits, at the final position.'''
    L, V = cfg.parity_len, cfg.vocab
    seq = torch.randint(2, V, (batch, L), device=device, generator=gen)   # noise tokens
    bit_a = torch.randint(0, 2, (batch,), device=device, generator=gen)
    bit_b = torch.randint(0, 2, (batch,), device=device, generator=gen)
    pos_a = torch.randint(0, L//3, (batch,), device=device, generator=gen)   # early third
    pos_b = torch.randint(2*L//3, L, (batch,), device=device, generator=gen) # late third
    bidx = torch.arange(batch, device=device)
    seq[bidx, pos_a] = bit_a
    seq[bidx, pos_b] = bit_b
    tgt = torch.full((batch, L), -100, device=device)
    tgt[:, -1] = (bit_a ^ bit_b)
    return seq, tgt


TASKS = {
    "copy":           dict(gen=make_copy_batch,           vocab_extra=2, readout=None, mode="seq"),
    "selective_copy": dict(gen=make_selective_copy_batch, vocab_extra=3, readout=None, mode="seq"),
    "assoc_recall":   dict(gen=make_assoc_recall_batch,   vocab_extra=1, readout=None, mode="final"),
    "parity":         dict(gen=make_parity_batch,         vocab_extra=0, readout=2,    mode="final"),
}

def task_dims(task, cfg):
    spec = TASKS[task]
    vocab_size = cfg.vocab + spec["vocab_extra"]
    readout_size = spec["readout"] if spec["readout"] is not None else vocab_size
    return vocab_size, readout_size

def task_chance(task, cfg):
    if task == "parity": return 0.5
    return 1.0 / cfg.vocab   # copy/selective/recall: uniform over vocab content tokens

## 4. Trainer

One trainer for all (arm, task) combinations. `chaossm_corridor` engages dual ascent toward
`corridor_target` after `warmup_frac`; `linear` and `chaossm_free` never apply a Lyapunov
penalty (the latter is the 0d "free" arm, carried forward unconstrained since it tied the best
explicit target).

In [6]:
def train_one(arm, task, seed, cfg):
    torch.manual_seed(seed); np.random.seed(seed)
    gen = torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gen.manual_seed(seed)
    vocab_size, readout_size = task_dims(task, cfg)
    phi_kind = "linear" if arm == "linear" else "chaossm"
    model = ChaoSSMLM(cfg, phi_kind, vocab_size, readout_size).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, betas=(0.9, 0.95))
    batch_fn = TASKS[task]["gen"]

    use_lyap = (arm == "chaossm_corridor")
    lo = hi = None
    if use_lyap:
        lo = cfg.corridor_target - cfg.corridor_half_width
        hi = cfg.corridor_target + cfg.corridor_half_width
    a_hi = a_lo = 0.0
    warmup_steps = int(cfg.warmup_frac * cfg.steps)
    hist, diverged = [], False

    for step in range(cfg.steps):
        inp, tgt = batch_fn(cfg, cfg.batch, DEVICE, gen)
        need_traj = (arm != "linear")
        out = model(inp, return_traj=need_traj)
        logits, traj, xseq = out if need_traj else (out, None, None)
        L_task = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), tgt.reshape(-1),
                                 ignore_index=-100)

        L_pen = torch.zeros((), device=DEVICE); lam_val = float("nan")
        if arm != "linear":
            in_warmup = step < warmup_steps
            lam = lambda_window(model, traj, xseq, cfg.lyap_window, cfg.lyap_subsample)
            lam_val = float(lam.detach())
            if use_lyap and not in_warmup:
                L_pen = a_hi * F.relu(lam - hi) + a_lo * F.relu(lo - lam)

        opt.zero_grad(set_to_none=True); (L_task + L_pen).backward()
        gn = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip).item()
        opt.step()

        if use_lyap and step >= warmup_steps:
            a_hi = float(np.clip(a_hi + cfg.dual_lr * (lam_val - hi), 0.0, cfg.alpha_max))
            a_lo = float(np.clip(a_lo + cfg.dual_lr * (lo - lam_val), 0.0, cfg.alpha_max))

        if not (math.isfinite(L_task.item()) and math.isfinite(gn)):
            diverged = True; break

        if step % cfg.eval_every == 0:
            with torch.no_grad():
                mask = tgt != -100
                pred = logits.argmax(-1)
                acc = ((pred == tgt) & mask).float().sum().item() / mask.float().sum().clamp(min=1).item()
            hist.append(dict(step=step, arm=arm, task=task, seed=seed,
                             loss=L_task.item(), acc=acc, lam=lam_val, gnorm=gn))

    tail = hist[-max(1, len(hist)//5):]
    s = dict(arm=arm, task=task, seed=seed, diverged=diverged,
             final_acc=float(np.mean([h["acc"] for h in tail])) if tail else np.nan,
             best_acc=float(np.max([h["acc"] for h in hist])) if hist else np.nan,
             lam_settled=float(np.nanmean([h["lam"] for h in tail])) if tail else np.nan,
             gnorm_max=float(np.max([h["gnorm"] for h in hist])) if hist else np.nan)
    return s, pd.DataFrame(hist)

## 5. Run — all arms × tasks × seeds

In [7]:
rows, hists = [], []
total = len(CFG.arms) * len(CFG.tasks) * CFG.seeds
done = 0
for task in CFG.tasks:
    chance = task_chance(task, CFG)
    for arm in CFG.arms:
        for sd in range(CFG.seeds):
            s, h = train_one(arm, task, sd, CFG)
            rows.append(s); hists.append(h); done += 1
            flag = " DIVERGED" if s["diverged"] else ""
            print(f"[{done}/{total}] {task:15s} {arm:18s} seed={sd}  "
                  f"final={s['final_acc']:.3f} (chance {chance:.3f})  "
                  f"best={s['best_acc']:.3f}  lam={s['lam_settled']:+.3f}{flag}")

SUM = pd.DataFrame(rows); HIST = pd.concat(hists, ignore_index=True)
savetable(SUM, "phase1_summary.csv"); savetable(HIST, "phase1_history.csv")

[1/60] copy            linear             seed=0  final=0.062 (chance 0.062)  best=0.085  lam=+nan
[2/60] copy            linear             seed=1  final=0.056 (chance 0.062)  best=0.089  lam=+nan
[3/60] copy            linear             seed=2  final=0.087 (chance 0.062)  best=0.108  lam=+nan
[4/60] copy            linear             seed=3  final=0.065 (chance 0.062)  best=0.090  lam=+nan
[5/60] copy            linear             seed=4  final=0.063 (chance 0.062)  best=0.082  lam=+nan
[6/60] copy            chaossm_free       seed=0  final=0.149 (chance 0.062)  best=0.176  lam=+0.340
[7/60] copy            chaossm_free       seed=1  final=0.149 (chance 0.062)  best=0.173  lam=+0.326
[8/60] copy            chaossm_free       seed=2  final=0.148 (chance 0.062)  best=0.167  lam=+0.306
[9/60] copy            chaossm_free       seed=3  final=0.151 (chance 0.062)  best=0.172  lam=+0.352
[10/60] copy            chaossm_free       seed=4  final=0.098 (chance 0.062)  best=0.138  lam=+0.445

## 6. Results table and pairwise comparisons

Two comparisons per task: **`chaossm_free` vs `linear`** (does chaos beat the linear ceiling?
— the central novelty question) and **`chaossm_corridor` vs `chaossm_free`** (does explicit
control beat the model's own operating point? — the 0d question, now on real tasks).

In [8]:
piv = SUM.pivot_table(index="task", columns="arm", values="final_acc", aggfunc="mean")
piv["chance"] = [task_chance(t, CFG) for t in piv.index]
print(piv.round(4).to_string())
savetable(piv.reset_index(), "phase1_results_table.csv")

def paired(task, a, b, col="final_acc"):
    x = SUM[(SUM.task==task)&(SUM.arm==a)].sort_values("seed")[col].values
    y = SUM[(SUM.task==task)&(SUM.arm==b)].sort_values("seed")[col].values
    m = np.isfinite(x) & np.isfinite(y); x, y = x[m], y[m]
    if len(x) < 3 or np.allclose(x, y):
        return dict(task=task, a=a, b=b, n=len(x), p=np.nan,
                    median_diff=float(np.median(x-y)) if len(x) else np.nan)
    st, p = wilcoxon(x, y)
    return dict(task=task, a=a, b=b, n=len(x), p=float(p), median_diff=float(np.median(x-y)))

rows = []
for task in CFG.tasks:
    rows.append(paired(task, "chaossm_free", "linear"))
    rows.append(paired(task, "chaossm_corridor", "chaossm_free"))
ST = pd.DataFrame(rows)
print()
print(ST.to_string(index=False))
savetable(ST, "phase1_stats.csv")
print(f"\nmin achievable two-sided p at n={CFG.seeds}: {2/(2**CFG.seeds):.4f}")

arm             chaossm_corridor  chaossm_free  linear  chance
task                                                          
assoc_recall              0.0930        0.0893  0.1547  0.0625
copy                      0.1544        0.1392  0.0666  0.0625
parity                    0.4878        0.5047  0.4927  0.5000
selective_copy            0.1036        0.0928  0.0680  0.0625
saved /content/drive/My Drive/ChaoSSM/results/phase1/phase1_results_table.csv

          task                a            b  n      p  median_diff
          copy     chaossm_free       linear  5 0.0625     0.086263
          copy chaossm_corridor chaossm_free  5 0.0625     0.013129
selective_copy     chaossm_free       linear  5 0.1250     0.010851
selective_copy chaossm_corridor chaossm_free  5 0.4375     0.012153
  assoc_recall     chaossm_free       linear  5 0.0625    -0.063802
  assoc_recall chaossm_corridor chaossm_free  5 0.8125     0.006510
        parity     chaossm_free       linear  5 0.3125     0.006510

In [9]:
fig, ax = plt.subplots(1, len(CFG.tasks), figsize=(5*len(CFG.tasks), 4.2), sharey=False)
if len(CFG.tasks) == 1: ax = [ax]
for i, task in enumerate(CFG.tasks):
    d = SUM[SUM.task == task]
    means = d.groupby("arm").final_acc.mean().reindex(CFG.arms)
    sds = d.groupby("arm").final_acc.std().reindex(CFG.arms)
    ax[i].bar(range(len(CFG.arms)), means.values, yerr=sds.values, capsize=4,
             color=["gray","tab:blue","tab:green"])
    ax[i].axhline(task_chance(task, CFG), color="r", ls="--", label="chance")
    ax[i].set_xticks(range(len(CFG.arms))); ax[i].set_xticklabels(CFG.arms, rotation=20, ha="right")
    ax[i].set(title=task, ylabel="final accuracy" if i==0 else "")
    ax[i].legend(fontsize=7); ax[i].grid(alpha=.3, axis="y")
fig.tight_layout(); savefig(fig, "phase1_results.png"); plt.close(fig)

saved /content/drive/My Drive/ChaoSSM/results/phase1/phase1_results.png


## 7. Verdict

| Observation | Reading |
|---|---|
| `chaossm_free` beats `linear` broadly, significant on the long-range tasks especially | Direct support for the proposal's central novelty claim — chaos exceeds the linear ceiling on the tasks that matter. |
| `chaossm_free` ≈ `linear` everywhere | The chaotic mechanism is controllable (0a–0d) but not yet shown to help on tasks; the expressivity claim needs a harder task or a different comparison. |
| `chaossm_free` beats `linear` only on `parity` (the long-range task) | The most interpretable positive result: chaos helps specifically where long-range integration matters, not on tasks solvable by local pattern matching — exactly what §3.2's fading-memory argument predicts. |
| `chaossm_corridor` ≈ `chaossm_free` on every task | Consistent with 0d: the model's own operating point is already about as good as an explicitly enforced one. Simplifies the architecture story — corridor control may not be necessary in practice. |
| `chaossm_corridor` beats `chaossm_free` on a specific task | The explicit constraint earns its complexity there; report which task and why. |
| `linear` beats both chaotic arms anywhere | An honest negative result for that task — report it plainly rather than omitting it. |

Whatever the pattern, report every task's numbers, not just the favorable ones — a
Q1 reviewer will ask for exactly this table.

In [10]:
verdict = dict(
    smoke_test=SMOKE_TEST,
    results_table=piv.round(4).to_dict("index"),
    stats=ST.to_dict("records"),
    config=dict(steps=CFG.steps, seeds=CFG.seeds, arms=CFG.arms, tasks=CFG.tasks,
                corridor_target=CFG.corridor_target),
)
savejson(verdict, "phase1_verdict.json")
print(json.dumps(verdict, indent=2, default=str))

saved /content/drive/My Drive/ChaoSSM/results/phase1/phase1_verdict.json
{
  "smoke_test": false,
  "results_table": {
    "assoc_recall": {
      "chaossm_corridor": 0.093,
      "chaossm_free": 0.0893,
      "linear": 0.1547,
      "chance": 0.0625
    },
    "copy": {
      "chaossm_corridor": 0.1544,
      "chaossm_free": 0.1392,
      "linear": 0.0666,
      "chance": 0.0625
    },
    "parity": {
      "chaossm_corridor": 0.4878,
      "chaossm_free": 0.5047,
      "linear": 0.4927,
      "chance": 0.5
    },
    "selective_copy": {
      "chaossm_corridor": 0.1036,
      "chaossm_free": 0.0928,
      "linear": 0.068,
      "chance": 0.0625
    }
  },
  "stats": [
    {
      "task": "copy",
      "a": "chaossm_free",
      "b": "linear",
      "n": 5,
      "p": 0.0625,
      "median_diff": 0.08626302083333333
    },
    {
      "task": "copy",
      "a": "chaossm_corridor",
      "b": "chaossm_free",
      "n": 5,
      "p": 0.0625,
      "median_diff": 0.01312934027777779
    